# Regresja logistyczna — przykład 2  
## Czy klient kupi usługę premium?

Ten notebook pokazuje regresję logistyczną bardziej **biznesowo i interpretacyjnie**.

Będziemy przewidywać, czy klient kupi usługę premium na podstawie:
- wieku,
- liczby wizyt na stronie,
- czasu spędzonego w serwisie,
- informacji, czy klient otrzymał ofertę rabatową.

Dane są syntetyczne, więc notebook działa bez pobierania plików z Internetu.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

np.random.seed(42)


## 1. Generujemy przykładowe dane

W prawdziwym projekcie ten fragment zastąpiłby import danych z CRM, CSV lub bazy danych.


In [ ]:
n = 1200

wiek = np.random.randint(18, 70, n)
wizyty = np.random.poisson(5, n)
czas_min = np.random.gamma(shape=2.2, scale=4.0, size=n)
rabat = np.random.binomial(1, 0.35, n)

# ukryty mechanizm generujący prawdopodobieństwo zakupu
logit = (
    -5.2
    + 0.035 * wiek
    + 0.22 * wizyty
    + 0.09 * czas_min
    + 1.10 * rabat
)

p = 1 / (1 + np.exp(-logit))
kupil = np.random.binomial(1, p)

df = pd.DataFrame({
    "wiek": wiek,
    "wizyty": wizyty,
    "czas_min": czas_min.round(2),
    "rabat": rabat,
    "kupil": kupil
})

df.head()


In [ ]:
print("Rozmiar danych:", df.shape)
print("\nOdsetek klientów, którzy kupili:")
print(df["kupil"].value_counts(normalize=True).round(3))


## 2. Szybka eksploracja

Sprawdzamy średnie wartości cech w grupie klientów, którzy kupili i nie kupili.


In [ ]:
df.groupby("kupil")[["wiek", "wizyty", "czas_min", "rabat"]].mean().round(2)


## 3. Przygotowanie danych


In [ ]:
X = df[["wiek", "wizyty", "czas_min", "rabat"]]
y = df["kupil"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Model regresji logistycznej


In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)


## 5. Ocena modelu


In [ ]:
print("AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("\nMacierz pomyłek:")
print(confusion_matrix(y_test, y_pred))

print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred))


## 6. Interpretacja współczynników

Po standaryzacji możemy porównywać względną siłę wpływu cech.

- dodatni współczynnik → cecha zwiększa prawdopodobieństwo klasy `1`,
- ujemny współczynnik → cecha zmniejsza prawdopodobieństwo klasy `1`.


In [ ]:
coef = pd.DataFrame({
    "cecha": X.columns,
    "wspolczynnik": model.coef_[0]
}).sort_values("wspolczynnik", ascending=False)

coef


## 7. Co się stanie, jeśli zmienimy próg decyzyjny?

Domyślnie `predict()` używa progu około `0.5`.

W biznesie możemy jednak świadomie użyć innego progu:
- niższy próg → więcej klientów zaklasyfikowanych jako potencjalnie zainteresowani,
- wyższy próg → mniej klientów, ale większa pewność.


In [ ]:
for threshold in [0.30, 0.50, 0.70]:
    pred_threshold = (y_prob >= threshold).astype(int)

    print(f"\nPRÓG = {threshold}")
    print(confusion_matrix(y_test, pred_threshold))


## 8. Przewidywanie dla pojedynczego klienta


In [ ]:
nowy_klient = pd.DataFrame([{
    "wiek": 44,
    "wizyty": 8,
    "czas_min": 16.5,
    "rabat": 1
}])

nowy_klient_scaled = scaler.transform(nowy_klient)

p_zakupu = model.predict_proba(nowy_klient_scaled)[0, 1]

print(f"Szacowane prawdopodobieństwo zakupu: {p_zakupu:.1%}")


## 9. Jak próg zmienia liczbę wybranych klientów?


In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
selected = [(y_prob >= t).sum() for t in thresholds]

plt.figure(figsize=(7, 5))
plt.plot(thresholds, selected, marker="o")
plt.xlabel("Próg decyzyjny")
plt.ylabel("Liczba klientów zaklasyfikowanych jako 1")
plt.title("Wpływ progu na liczbę wybranych klientów")
plt.grid(alpha=0.3)
plt.show()


## Pytania do grupy

1. Dlaczego model zwraca prawdopodobieństwo, a nie tylko klasę?
2. Kiedy warto obniżyć próg z 0.5 do np. 0.3?
3. Co jest bardziej kosztowne: false positive czy false negative?
4. Jak wykorzystać regresję logistyczną w marketingu, HR, bankowości albo medycynie?
